<a href="https://colab.research.google.com/github/teguhsmlnna666/BigData26_B_2411532005_TeguhEsaMaulanna/blob/main/Praktikum2/BD_B_P02_2411532005_TeguhEsaMaulanna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langkah ini untuk menginstal library Faker yang digunakan untuk menghasilkan data sintetis atau data dummy.

In [ ]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.9 MB/s eta 0:00:00


Langkah ini untuk mengimpor library NumPy, Pandas, Faker, dan Random yang digunakan untuk pembuatan, pengolahan, dan pengacakan data transaksi sintetis.

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
import random

Langkah ini untuk membuat dataset transaksi sintetis sebanyak 500 baris dengan berbagai atribut seperti pelanggan, produk, kategori, harga, jumlah, metode pembayaran, tanggal, kota, dan rating. Dataset juga diberikan variasi format untuk mensimulasikan kondisi data mentah. Lalu ditambahkan nilai missing value dan 15 baris data duplikat sebagai simulasi data asli.

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar}",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper()

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


Langkah ini untuk mendeteksi jumlah missing value pada setiap kolom dan melakukan penanganan dengan menghapus data yang kehilangan informasi wajib serta mengisi kota pengiriman yang kosong dengan nilai "Tidak Diketahui".

In [ ]:
# Deteksi missing value
print(df.isnull().sum())

# Penanganan missing value
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

print("Jumlah baris setelah missing value():", len(df))

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64
Jumlah baris setelah missing value(): 495


Langkah ini untuk mendeteksi data duplikat pada dataset dan menghapus baris yang memiliki seluruh nilai kolom yang sama agar setiap transaksi tidak tercatat lebih dari satu kali.

In [ ]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'])

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 0      TRX00305
1      TRX00500
2      TRX00442
3      TRX00154
5      TRX00132
         ...   
510    TRX00072
511    TRX00107
512    TRX00271
513    TRX00436
514    TRX00103
Name: transaction_id, Length: 495, dtype: object
Jumlah baris setelah drop_duplicates(): 490


Langkah ini untuk menyamakan format teks pada kolom kategori, metode pembayaran, dan kota pengiriman dengan menghapus spasi berlebih serta menyamakan penggunaan huruf. Penulisan COD juga dikembalikan ke format kapital.

In [ ]:
# standarisasi teks kategorikal

for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

Langkah ini untuk membersihkan variasi format pada kolom harga dengan menghapus simbol mata uang, tanda baca, dan spasi, kemudian mengubah nilai harga menjadi tipe data numerik.

In [ ]:
# koreksi tipe data pada kolom price

def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

Langkah ini untuk menyamakan berbagai format tanggal pada kolom transaction_date menjadi format YYYY-MM-DD.

In [ ]:
# Standarisasi format tanggal

def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

Langkah ini untuk memastikan tipe data pada kolom quantity dan price sesuai dengan karakteristik datanya, yaitu integer untuk jumlah barang dan float untuk nilai harga.

In [ ]:
# finalisasi tipe data

df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

Langkah ini untuk menyimpan dataset yang telah melalui seluruh proses cleaning ke dalam file transaksi_bersih.csv serta memeriksa jumlah baris data setelah proses pembersihan selesai.

In [ ]:
# dataset bersih

df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


Langkah ini digunakan untuk menyimpan file hasil analisis ke dalam google drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA   = "/content/data"
DIR_SIMPAN  ="/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Mounted at /content/drive
['transaksi_bersih.csv', 'transaksi_mentah.csv', 'transaksi_mentah_latihan.csv', 'transaksi_bersih_latihan.csv']


In [ ]:
path_mentah = os.path.join(DIR_SIMPAN, "transaksi_mentah.csv")
df.to_csv(path_mentah, index=False)

file_path = os.path.join(DIR_SIMPAN, 'transaksi_bersih.csv')
df.to_csv(file_path, index=False)

Studi Kasus



1. Mengapa total transaksi yang dilaporkan tim IT (515) berbeda dengan total yang digunakan tim Finance (490)?<br>
Perbedaan ini terjadi karena kedua tim menggunakan data pada tahapan yang berbeda. Tim IT melaporkan jumlah 515 baris data mentah, yaitu 500 transaksi awal ditambah 15 baris duplikat. Setelah proses cleaning, dilakukan penghapusan 15 data duplikat dan 10 baris dengan customer_name atau payment_method yang kosong. Dengan demikian, jumlah data yang digunakan setelah preprocessing menjadi 490 baris. Perbedaan 25 baris tersebut berasal dari penerapan aturan kualitas data, bukan karena adanya kesalahan dalam perhitungan jumlah baris.
2. Apakah 490 baris “lebih benar” dibandingkan 515 baris? Kaitkan dengan konsep Veracity.<br>
Nilai 490 baris tidak dapat dikatakan mutlak lebih benar daripada 515 karena keduanya menggambarkan kondisi data pada tahap yang berbeda. Sebanyak 515 baris merupakan jumlah data mentah sebelum proses cleaning, sedangkan 490 baris merupakan jumlah data setelah mengatasi data duplikat dan data dengan informasi wajib yang hilang. Dalam konsep Veracity, kualitas dan keandalan data perlu diperhatikan agar data yang digunakan dalam analisis dapat merepresentasikan kondisi yang sebenarnya. Oleh karena itu, angka 490 lebih sesuai digunakan untuk analisis transaksi setelah aturan preprocessing diterapkan, sedangkan angka 515 tetap penting sebagai jumlah awal data mentah dan sebagai bagian dari dokumentasi proses.
3. Bagaimana menjelaskan keputusan membiarkan kolom rating tetap memiliki missing value kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”?<br>
Kolom rating sengaja tidak diisi karena tidak adanya informasi yang cukup untuk mengetahui rating sebenarnya dari transaksi yang kosong. Mengisi nilai tersebut dengan angka perkiraan dapat mengubah distribusi rating dan memengaruhi hasil rata-rata. Oleh karena itu, untuk menghitung rating rata-rata, Finance sebaiknya menggunakan transaksi yang memiliki rating saja. Selain nilai rata-rata, jumlah transaksi yang memiliki rating dan jumlah rating yang kosong juga perlu dilaporkan agar hasil analisis memiliki konteks yang jelas.

